In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("../data/marketing_and_sales.csv")

Avant de procéder à la modélisation, nous devons traiter les valeurs manquantes de notre variable cible (`Sales`) en supprimant les lignes concernées. 
Ensuite, nous divisons notre jeu de données en un ensemble d'entraînement (80%) et un ensemble de test (20%). Cette séparation stricte est indispensable pour évaluer les performances de nos modèles sur des données totalement inédites et éviter toute fuite de données (data leakage).

In [2]:
df = df.dropna(subset=['Sales'])

X = df.drop('Sales', axis=1)
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

L'EDA a mis en évidence la présence de valeurs manquantes et d'outliers dans certaines de nos variables (Radio et Social Media). 
Pour un traitement propre, nous mettons en place un ColumnTransformer :
- **Variables numériques** (`TV`, `Radio`, `Social Media`, `Budget total`) : Imputation des valeurs manquantes par la médiane
- **Variable string** (`Influencer`) : Imputation par la valeur la plus fréquente puis encodage.

In [6]:
num_features = ['TV', 'Radio', 'Social Media']
cat_features = ['Influencer']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

Pour répondre aux exigences du projet, nous allons implémenter et comparer trois algorithmes différents, du plus simple au plus complexe :
1. Régression Linéaire : Servira de modèle de base, simple et facilement interprétable.
2. Random Forest Regressor : Modèle robuste capable de capturer des relations non linéaires.
3. Gradient Boosting Regressor : Algorithme avancé qui corrige séquentiellement les erreurs des arbres de décision précédents.

In [8]:
pipeline_lr = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', LinearRegression())])

pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', RandomForestRegressor(random_state=42))])

pipeline_gb = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', GradientBoostingRegressor(random_state=42))])

In [9]:
pipeline_lr.fit(X_train, y_train)
pipeline_rf.fit(X_train, y_train)
pipeline_gb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

Nous utiliserons deux métriques adaptées à la régression :
- **RMSE (Root Mean Squared Error)** : Indique l'erreur moyenne de prédiction dans la même unité que les ventes. On cherche à le minimiser.
- **R² (Coefficient de détermination)** : Indique la proportion de la variance des ventes expliquée par le modèle. On cherche à s'approcher de 1.

In [10]:
def evaluate_model(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}\n")

evaluate_model("Régression Linéaire", pipeline_lr, X_test, y_test)
evaluate_model("Random Forest", pipeline_rf, X_test, y_test)
evaluate_model("Gradient Boosting", pipeline_gb, X_test, y_test)

Régression Linéaire
RMSE : 5.88
R²   : 0.9960

Random Forest
RMSE : 3.80
R²   : 0.9983

Gradient Boosting
RMSE : 3.31
R²   : 0.9987



In [11]:
def evaluate_model_cv(name, pipeline, X_train, y_train):
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    print(f"{name}")
    print(f"{scores}")
    print(f"R² moyen : {scores.mean():.4f} (+/- {scores.std() * 2:.4f})\n")

evaluate_model_cv("Régression Linéaire", pipeline_lr, X_train, y_train)
evaluate_model_cv("Random Forest", pipeline_rf, X_train, y_train)
evaluate_model_cv("Gradient Boosting", pipeline_gb, X_train, y_train)

Régression Linéaire
[0.99167279 0.99361329 0.99788392 0.99618806 0.99562029]
R² moyen : 0.9950 (+/- 0.0043)

Random Forest
[0.99538485 0.99228485 0.99757113 0.99727811 0.99557364]
R² moyen : 0.9956 (+/- 0.0038)

Gradient Boosting
[0.99173977 0.99182643 0.99783608 0.99616181 0.99472206]
R² moyen : 0.9945 (+/- 0.0048)



In [12]:
os.makedirs('../models', exist_ok=True)
joblib.dump(pipeline_gb, '../models/gradient_boosting_pipeline.pkl')

['../models/gradient_boosting_pipeline.pkl']

### Conclusion et choix du modèle

Les trois modèles affichent des performances avec des R² supérieurs à 99%. Ces scores reflètent la relation quasi-parfaite entre le budget TV et les Ventes.

Toutefois, les modèles ensemblistes surpassent la Régression Linéaire. Le **Gradient Boosting** parvient à réduire l'erreur moyenne (RMSE) à **3.31** tout en expliquant **99.87%** de la variance. 

**Choix du modèle :** Gradient Boosting 